<a href="https://colab.research.google.com/github/NamishBansal15/substation-detection/blob/main/model-training/rfdetr_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install roboflow

In [ ]:
import os
import json
import shutil
import threading
import time
import random
from pathlib import Path
from datetime import datetime
from getpass import getpass
import wandb
from roboflow import Roboflow
from google.colab import drive

In [ ]:
# ── Drive mount ─────────────────────────────────────────────────
drive.mount('/content/drive')
DRIVE_DIR = os.environ.get("RFDETR_DRIVE_DIR", "/content/drive/MyDrive/april-detection/rfdetr")
os.makedirs(DRIVE_DIR, exist_ok=True)

# ── Dataset download ─────────────────────────────────────────────
rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("space-weather").project("merged-dataset-gzbkg")
dataset = project.version(2).download("coco")
COCO_DIR = dataset.location
print(f"Dataset ready at: {COCO_DIR}")

# ── COCO split ───────────────────────────────────────────────────
def split_coco_dataset(dataset_location, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, seed=42):
    random.seed(seed)

    src_ann = Path(dataset_location) / "train" / "_annotations.coco.json"

    with open(src_ann) as f:
        coco = json.load(f)

    for split in ["valid", "test"]:
        (Path(dataset_location) / split).mkdir(parents=True, exist_ok=True)

    images = coco["images"].copy()
    random.shuffle(images)

    n       = len(images)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train_imgs = images[:n_train]
    val_imgs   = images[n_train:n_train + n_val]
    test_imgs  = images[n_train + n_val:]

    print(f"Total: {n} → Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

    def make_coco_split(img_list, split_name):
        img_ids    = {img["id"] for img in img_list}
        split_anns = [a for a in coco["annotations"] if a["image_id"] in img_ids]
        split_coco = {
            "info":        coco.get("info", {}),
            "licenses":    coco.get("licenses", []),
            "categories":  coco["categories"],
            "images":      img_list,
            "annotations": split_anns,
        }
        ann_out = Path(dataset_location) / split_name / "_annotations.coco.json"
        with open(ann_out, "w") as f:
            json.dump(split_coco, f, indent=2)

        if split_name != "train":
            for img in img_list:
                src = Path(dataset_location) / "train" / img["file_name"]
                dst = Path(dataset_location) / split_name / img["file_name"]
                if src.exists() and src.resolve() != dst.resolve():
                    shutil.move(str(src), str(dst))

        print(f"  {split_name}: {len(img_list)} images, {len(split_anns)} annotations")

    make_coco_split(train_imgs, "train")
    make_coco_split(val_imgs,   "valid")
    make_coco_split(test_imgs,  "test")
    print("✅ Split complete!")

print("Splitting dataset 70/20/10...")
split_coco_dataset(COCO_DIR)

# ── Config ───────────────────────────────────────────────────────
WANDB_PROJECT = "april-detection"

configs = [
    {"name": "rfdetr-base",  "variant": "base",  "epochs": 50, "batch": 4, "lr": 1e-4},
    {"name": "rfdetr-large", "variant": "large", "epochs": 50, "batch": 2, "lr": 1e-4},
]

# ── Checkpoint sync ──────────────────────────────────────────────
def sync_checkpoints(run_name, stop_event):
    src = f"/content/{run_name}"
    dst = f"{DRIVE_DIR}/{run_name}"
    os.makedirs(dst, exist_ok=True)
    while not stop_event.is_set():
        try:
            if os.path.exists(src):
                for f in os.listdir(src):
                    if f.endswith(".pth"):
                        s = f"{src}/{f}"
                        d = f"{dst}/{f}"
                        if not os.path.exists(d):
                            shutil.copy2(s, d)
                            print(f"[Sync] Saved {run_name}/{f} to Drive")
        except Exception as e:
            print(f"[Sync] Error: {e}")
        time.sleep(300)

# ── Resume helper ────────────────────────────────────────────────
def find_latest_checkpoint(run_name):
    dst = f"{DRIVE_DIR}/{run_name}"
    if not os.path.exists(dst):
        return None
    checkpoints = sorted([
        f for f in os.listdir(dst)
        if f.startswith("checkpoint") and f.endswith(".pth")
    ])
    if checkpoints:
        path = f"{dst}/{checkpoints[-1]}"
        print(f"[Resume] Found: {checkpoints[-1]}")
        return path
    return None

# ── Training loop ────────────────────────────────────────────────
wandb.login()

from rfdetr import RFDETRBase, RFDETRLarge

for cfg in configs:
    print(f"\n{'='*50}")
    print(f"Starting: {cfg['name']}")
    print(f"{'='*50}")

    output_dir = f"/content/{cfg['name']}"
    os.makedirs(output_dir, exist_ok=True)

    resume_path = find_latest_checkpoint(cfg["name"])

    # Start Drive sync thread
    stop_event  = threading.Event()
    sync_thread = threading.Thread(
        target=sync_checkpoints,
        args=(cfg["name"], stop_event),
        daemon=True
    )
    sync_thread.start()

    wandb.init(
        project=WANDB_PROJECT,
        name=cfg["name"],
        config=cfg,
        reinit=True
    )

    model = RFDETRBase() if cfg["variant"] == "base" else RFDETRLarge()

    model.train(
        dataset_dir=COCO_DIR,
        epochs=cfg["epochs"],
        batch_size=cfg["batch"],
        lr=cfg["lr"],
        output_dir=output_dir,
        checkpoint_interval=5,
        early_stopping=True,
        early_stopping_patience=10,
        wandb=True,
        project=WANDB_PROJECT,
        run=cfg["name"],
        resume=resume_path,
    )

    stop_event.set()

    # Copy final checkpoint to Drive
    best_path = f"{output_dir}/checkpoint_best_total.pth"
    if os.path.exists(best_path):
        dst = f"{DRIVE_DIR}/{cfg['name']}/checkpoint_best_total.pth"
        os.makedirs(f"{DRIVE_DIR}/{cfg['name']}", exist_ok=True)
        shutil.copy2(best_path, dst)
        print(f"[Drive] Final checkpoint saved")

    wandb.finish()
    print(f"✅ Finished: {cfg['name']}")

print("\n🎉 All runs complete!")